In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait
from bs4 import BeautifulSoup
import pandas as pd

# Chrome 드라이버 세팅
service = Service(executable_path=ChromeDriverManager().install())
driver = webdriver.Chrome(service=service)

# URL 열기
url = 'https://www.jobkorea.co.kr/Search/?stext=%EB%8D%B0%EC%9D%B4%ED%84%B0%EB%B6%84%EC%84%9D&tabType=recruit&Page_No=1'
driver.get(url)

# 페이지 소스 가져오기
wait = WebDriverWait(driver, 10)
html = driver.page_source
driver.quit()

# BeautifulSoup로 HTML 파싱
soup = BeautifulSoup(html, 'html.parser')

# jobkor_contents 추출
jobkor_contents = soup.find_all('article', class_='list')

# 데이터를 저장할 리스트
jobkor = []

# 기본 URL
base_url = "https://www.jobkorea.co.kr"

# jobkor_contents에서 데이터 추출
for i in jobkor_contents:
    jobkor_list = i.find_all('article', class_='list-item')

    for i in jobkor_list:
        jobkor_info = {}

        # 회사명 추출
        jobkor_corp_tag = i.find('a', class_='corp-name-link')
        jobkor_corp = jobkor_corp_tag.get('title')

        # 모집 직무 추출
        jobkor_recruit_tag = i.find('a', class_='information-title-link')
        jobkor_recruit = jobkor_recruit_tag.text.strip()

        # 상세 정보 추출 (줄바꿈 및 불필요한 공백 제거)
        jobkor_detail_tag = i.find('ul', class_='chip-information-group')
        if jobkor_detail_tag:
            jobkor_detail = jobkor_detail_tag.text.strip().replace("\n", " ").replace("\r", "").strip()
        else:
            jobkor_detail = ""

        # 채용 공고 URL 추출
        jobkor_url_tag = i.find('a', class_='information-title-link')
        jobkor_url = jobkor_url_tag['href']
        full_url = base_url + jobkor_url

        # jobkor_info 딕셔너리 초기화
        jobkor_info['Site'] = 'Job_Korea'
        jobkor_info['Col_Company'] = jobkor_corp
        jobkor_info['Col_Recruit'] = jobkor_recruit.strip()
        jobkor_info['Col_detail'] = [jobkor_detail]  # 리스트로 초기화
        jobkor_info['Col_url'] = full_url

        # 혜택 정보 추출 (줄바꿈 및 불필요한 공백 제거)
        jobkor_corp_tags = i.find('ul', class_='chip-benefit-group')
        if jobkor_corp_tags:
            jobkor_benefit_detail = jobkor_corp_tags.text.strip().replace("\n", " ").replace("\r", "").strip()
            jobkor_info['Col_detail'].append(jobkor_benefit_detail)  # 리스트에 추가

        # jobkor 리스트에 jobkor_info 추가
        jobkor.append(jobkor_info)

# DataFrame으로 변환
df_jobkor = pd.DataFrame(jobkor)
df_jobkor.to_csv('data_tmp/data_jobkorea.csv', index=False)

# 결과 확인
df_jobkor.head()


/Users/baejieun/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


,Site,Col_Company,Col_Recruit,Col_detail,Col_url
0,Job_Korea,더치트주식회사,데이터 애널리스트 / 데이터분석 전문가 / 통계 전문가 / Data Analyst (통,[경력무관 대졸↑ 정규직 외 서울 구로구 D-24],https://www.jobkorea.co.kr/Recruit/GI_Read/466...
1,Job_Korea,넛지헬스케어㈜,[캐시워크-병역특례] 데이터분석 담당 산업기능요원,"[경력무관 학력무관 병역특례 서울 강남구 D-7, #쾌적한 업무공간 #유연근무]",https://www.jobkorea.co.kr/Recruit/GI_Read/465...
2,Job_Korea,㈜메디젠휴먼케어,메디젠휴먼케어 데이터분석 담당 경력직 채용,[경력3년↑ 석사↑ 정규직 서울 송파구 D-41],https://www.jobkorea.co.kr/Recruit/GI_Read/465...
3,Job_Korea,넛지헬스케어㈜,[캐시워크] 데이터분석 담당 채용전환형 인턴,"[신입 대졸↑ 인턴 서울 강남구 D-10, #식사 지원 #쾌적한 업무공간 #휴가제도...",https://www.jobkorea.co.kr/Recruit/GI_Read/465...
4,Job_Korea,한패스㈜,[한패스(주)/데이터분석팀] Google Analytics 데이터분석가,[경력 학력무관 정규직 서울 성동구 D-6],https://www.jobkorea.co.kr/Recruit/GI_Read/465...
